# Lecture 6: Variational Autoencoders (VAEs) for Protein Sequences

This notebook demonstrates the fundamentals of VAEs applied to protein sequences:
1. Understanding the VAE architecture
2. Implementing the ELBO loss and reparameterization trick
3. Training a VAE on protein sequences
4. Generating new sequences and exploring the latent space

In [ ]:
# Install dependencies if needed
# !pip install torch numpy matplotlib scikit-learn seaborn

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import seaborn as sns

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Data Preparation

We'll work with protein sequences represented as integer indices (0-20 for 21 amino acid tokens including gap).

In [ ]:
# Amino acid vocabulary
AA_VOCAB = 'ACDEFGHIKLMNPQRSTVWY-'  # 20 standard AAs + gap
AA_TO_IDX = {aa: i for i, aa in enumerate(AA_VOCAB)}
IDX_TO_AA = {i: aa for i, aa in enumerate(AA_VOCAB)}
VOCAB_SIZE = len(AA_VOCAB)

print(f"Vocabulary size: {VOCAB_SIZE}")
print(f"Amino acid mapping: {AA_TO_IDX}")

In [ ]:
def encode_sequence(seq):
    """Convert amino acid sequence to integer indices"""
    return torch.tensor([AA_TO_IDX.get(aa, AA_TO_IDX['-']) for aa in seq])

def decode_sequence(indices):
    """Convert integer indices back to amino acid sequence"""
    return ''.join([IDX_TO_AA[int(idx)] for idx in indices])

# Test encoding/decoding
test_seq = "MKFLILLFNILCLFPVLAADNH"
encoded = encode_sequence(test_seq)
decoded = decode_sequence(encoded)
print(f"Original:  {test_seq}")
print(f"Encoded:   {encoded.tolist()}")
print(f"Decoded:   {decoded}")

In [ ]:
# Generate synthetic protein sequences for demonstration
# In practice, you would load real protein family sequences

def generate_synthetic_sequences(n_sequences, seq_length, n_families=3):
    """
    Generate synthetic protein sequences with family structure.
    Each family has a conserved motif pattern.
    """
    sequences = []
    labels = []
    
    # Define family motifs (conserved regions)
    motifs = [
        "GAVLI",   # Hydrophobic family
        "DENKR",   # Charged family
        "STYCN"    # Polar family
    ]
    
    for i in range(n_sequences):
        family = i % n_families
        motif = motifs[family]
        
        # Generate sequence with family-specific pattern
        seq = []
        for j in range(seq_length):
            if j % 5 == 0:  # Conserved positions
                aa = np.random.choice(list(motif))
            else:  # Variable positions
                aa = np.random.choice(list(AA_VOCAB[:-1]))  # Exclude gap
            seq.append(aa)
        
        sequences.append(''.join(seq))
        labels.append(family)
    
    return sequences, labels

# Generate training data
SEQ_LENGTH = 50
N_SEQUENCES = 1000

sequences, family_labels = generate_synthetic_sequences(N_SEQUENCES, SEQ_LENGTH)

print(f"Generated {len(sequences)} sequences of length {SEQ_LENGTH}")
print(f"\nExample sequences from each family:")
for family in range(3):
    idx = family_labels.index(family)
    print(f"  Family {family}: {sequences[idx][:30]}...")

In [ ]:
# Create PyTorch dataset
class ProteinDataset(Dataset):
    def __init__(self, sequences, labels=None):
        self.sequences = [encode_sequence(seq) for seq in sequences]
        self.labels = labels
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        if self.labels is not None:
            return self.sequences[idx], self.labels[idx]
        return self.sequences[idx]

# Split data
train_size = int(0.8 * len(sequences))
train_sequences = sequences[:train_size]
train_labels = family_labels[:train_size]
val_sequences = sequences[train_size:]
val_labels = family_labels[train_size:]

train_dataset = ProteinDataset(train_sequences, train_labels)
val_dataset = ProteinDataset(val_sequences, val_labels)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

## 2. VAE Architecture

Now let's build the VAE model with:
- **Encoder**: Maps sequences to latent distribution parameters ($\mu$, $\log\sigma^2$)
- **Reparameterization**: Samples from the latent distribution
- **Decoder**: Reconstructs sequences from latent codes

In [ ]:
class ProteinVAE(nn.Module):
    """
    Variational Autoencoder for protein sequences.
    
    Architecture:
    - Input: One-hot encoded sequences [batch, seq_len, vocab_size]
    - Encoder: MLP that outputs mu and logvar
    - Latent: Sample z using reparameterization trick
    - Decoder: MLP that outputs logits over amino acids
    """
    
    def __init__(self, seq_length, vocab_size, hidden_dim=256, latent_dim=32):
        super().__init__()
        self.seq_length = seq_length
        self.vocab_size = vocab_size
        self.latent_dim = latent_dim
        
        input_dim = seq_length * vocab_size
        
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        
        # Latent space parameters
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
        )
    
    def encode(self, x):
        """
        Encode input to latent distribution parameters.
        
        Args:
            x: Input sequence indices [batch, seq_len]
        
        Returns:
            mu: Mean of latent distribution [batch, latent_dim]
            logvar: Log variance of latent distribution [batch, latent_dim]
        """
        # One-hot encode
        x_onehot = F.one_hot(x, self.vocab_size).float()
        x_flat = x_onehot.view(x.size(0), -1)
        
        # Encode
        h = self.encoder(x_flat)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        
        return mu, logvar
    
    def reparameterize(self, mu, logvar):
        """
        Reparameterization trick: z = mu + sigma * epsilon
        
        This allows gradients to flow through the sampling operation.
        
        Args:
            mu: Mean of latent distribution
            logvar: Log variance of latent distribution
        
        Returns:
            z: Sampled latent vector
        """
        if self.training:
            std = torch.exp(0.5 * logvar)  # sigma = exp(log(sigma^2) / 2)
            eps = torch.randn_like(std)     # epsilon ~ N(0, I)
            return mu + eps * std
        else:
            # During evaluation, use mean
            return mu
    
    def decode(self, z):
        """
        Decode latent vector to sequence logits.
        
        Args:
            z: Latent vector [batch, latent_dim]
        
        Returns:
            logits: Reconstruction logits [batch, seq_len, vocab_size]
        """
        h = self.decoder(z)
        logits = h.view(-1, self.seq_length, self.vocab_size)
        return logits
    
    def forward(self, x):
        """
        Full forward pass.
        
        Returns:
            recon_logits: Reconstruction logits
            mu: Latent mean
            logvar: Latent log variance
        """
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon_logits = self.decode(z)
        return recon_logits, mu, logvar
    
    def sample(self, n_samples):
        """
        Generate new sequences by sampling from the prior.
        
        Args:
            n_samples: Number of sequences to generate
        
        Returns:
            Generated sequence indices [n_samples, seq_len]
        """
        z = torch.randn(n_samples, self.latent_dim).to(next(self.parameters()).device)
        logits = self.decode(z)
        samples = torch.argmax(logits, dim=-1)
        return samples

# Initialize model
model = ProteinVAE(
    seq_length=SEQ_LENGTH,
    vocab_size=VOCAB_SIZE,
    hidden_dim=256,
    latent_dim=16  # Small latent dimension for visualization
).to(device)

print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

## 3. Loss Function: ELBO

The VAE loss (negative ELBO) consists of:
1. **Reconstruction Loss**: Cross-entropy between input and reconstruction
2. **KL Divergence**: Regularizes the latent space to be close to N(0, I)

$$\mathcal{L} = -\mathbb{E}_{q(z|x)}[\log p(x|z)] + D_{KL}(q(z|x) \| p(z))$$

In [ ]:
def vae_loss(recon_logits, x, mu, logvar, beta=1.0):
    """
    Compute VAE loss (negative ELBO).
    
    Args:
        recon_logits: Reconstruction logits [batch, seq_len, vocab_size]
        x: Original input indices [batch, seq_len]
        mu: Latent mean [batch, latent_dim]
        logvar: Latent log variance [batch, latent_dim]
        beta: Weight for KL term (beta-VAE)
    
    Returns:
        total_loss: Combined loss
        recon_loss: Reconstruction loss only
        kl_loss: KL divergence only
    """
    batch_size = x.size(0)
    
    # Reconstruction loss (cross-entropy)
    # Reshape for cross_entropy: [batch * seq_len, vocab_size] and [batch * seq_len]
    recon_loss = F.cross_entropy(
        recon_logits.view(-1, recon_logits.size(-1)),
        x.view(-1),
        reduction='sum'
    ) / batch_size
    
    # KL divergence: D_KL(N(mu, sigma) || N(0, I))
    # = -0.5 * sum(1 + log(sigma^2) - mu^2 - sigma^2)
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / batch_size
    
    # Total loss
    total_loss = recon_loss + beta * kl_loss
    
    return total_loss, recon_loss, kl_loss

# Test the loss function
test_batch = next(iter(train_loader))[0].to(device)
recon, mu, logvar = model(test_batch)
total, recon, kl = vae_loss(recon, test_batch, mu, logvar)
print(f"Test losses - Total: {total:.4f}, Recon: {recon:.4f}, KL: {kl:.4f}")

## 4. Training Loop

In [ ]:
def train_epoch(model, train_loader, optimizer, beta=1.0):
    model.train()
    total_loss = 0
    total_recon = 0
    total_kl = 0
    
    for batch in train_loader:
        x = batch[0].to(device)
        
        optimizer.zero_grad()
        recon_logits, mu, logvar = model(x)
        loss, recon_loss, kl_loss = vae_loss(recon_logits, x, mu, logvar, beta)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        total_recon += recon_loss.item()
        total_kl += kl_loss.item()
    
    n_batches = len(train_loader)
    return total_loss / n_batches, total_recon / n_batches, total_kl / n_batches


def evaluate(model, val_loader, beta=1.0):
    model.eval()
    total_loss = 0
    total_recon = 0
    total_kl = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch in val_loader:
            x = batch[0].to(device)
            recon_logits, mu, logvar = model(x)
            loss, recon_loss, kl_loss = vae_loss(recon_logits, x, mu, logvar, beta)
            
            total_loss += loss.item()
            total_recon += recon_loss.item()
            total_kl += kl_loss.item()
            
            # Reconstruction accuracy
            preds = torch.argmax(recon_logits, dim=-1)
            correct += (preds == x).sum().item()
            total += x.numel()
    
    n_batches = len(val_loader)
    accuracy = correct / total
    return total_loss / n_batches, total_recon / n_batches, total_kl / n_batches, accuracy

In [ ]:
# Training configuration
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
n_epochs = 50
beta = 1.0  # Standard VAE; increase for beta-VAE

# Training history
history = {
    'train_loss': [], 'train_recon': [], 'train_kl': [],
    'val_loss': [], 'val_recon': [], 'val_kl': [], 'val_acc': []
}

print("Starting training...\n")
for epoch in range(n_epochs):
    # Train
    train_loss, train_recon, train_kl = train_epoch(model, train_loader, optimizer, beta)
    
    # Evaluate
    val_loss, val_recon, val_kl, val_acc = evaluate(model, val_loader, beta)
    
    # Store history
    history['train_loss'].append(train_loss)
    history['train_recon'].append(train_recon)
    history['train_kl'].append(train_kl)
    history['val_loss'].append(val_loss)
    history['val_recon'].append(val_recon)
    history['val_kl'].append(val_kl)
    history['val_acc'].append(val_acc)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{n_epochs}")
        print(f"  Train - Loss: {train_loss:.4f}, Recon: {train_recon:.4f}, KL: {train_kl:.4f}")
        print(f"  Val   - Loss: {val_loss:.4f}, Recon: {val_recon:.4f}, KL: {val_kl:.4f}, Acc: {val_acc:.4f}")
        print()

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Total loss
axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Validation')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Total Loss')
axes[0].set_title('VAE Loss (ELBO)')
axes[0].legend()

# Reconstruction loss
axes[1].plot(history['train_recon'], label='Train')
axes[1].plot(history['val_recon'], label='Validation')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Reconstruction Loss')
axes[1].set_title('Reconstruction Loss')
axes[1].legend()

# KL divergence
axes[2].plot(history['train_kl'], label='Train')
axes[2].plot(history['val_kl'], label='Validation')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('KL Divergence')
axes[2].set_title('KL Divergence')
axes[2].legend()

plt.tight_layout()
plt.show()

## 5. Latent Space Visualization

In [ ]:
def get_latent_representations(model, data_loader):
    """Extract latent representations for all sequences"""
    model.eval()
    latents = []
    labels = []
    
    with torch.no_grad():
        for batch in data_loader:
            x = batch[0].to(device)
            batch_labels = batch[1]
            
            mu, _ = model.encode(x)
            latents.append(mu.cpu().numpy())
            labels.extend(batch_labels.tolist())
    
    return np.vstack(latents), np.array(labels)

# Get latent representations
latents, labels = get_latent_representations(model, val_loader)
print(f"Latent representations shape: {latents.shape}")

In [ ]:
# Visualize with t-SNE
if latents.shape[1] > 2:
    tsne = TSNE(n_components=2, random_state=42, perplexity=30)
    latents_2d = tsne.fit_transform(latents)
else:
    latents_2d = latents

plt.figure(figsize=(10, 8))
scatter = plt.scatter(latents_2d[:, 0], latents_2d[:, 1], 
                      c=labels, cmap='viridis', alpha=0.7)
plt.colorbar(scatter, label='Protein Family')
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')
plt.title('VAE Latent Space (t-SNE visualization)\nColored by Protein Family')
plt.show()

## 6. Sequence Generation and Reconstruction

In [ ]:
# Generate new sequences
model.eval()
n_generated = 5
generated_indices = model.sample(n_generated)

print("Generated sequences from random latent samples:\n")
for i in range(n_generated):
    seq = decode_sequence(generated_indices[i].cpu())
    print(f"  {i+1}. {seq}")

In [ ]:
# Reconstruction quality
model.eval()

# Get a sample batch
sample_batch = next(iter(val_loader))
x = sample_batch[0][:5].to(device)

with torch.no_grad():
    recon_logits, _, _ = model(x)
    recon = torch.argmax(recon_logits, dim=-1)

print("Original vs Reconstructed sequences:\n")
for i in range(5):
    original = decode_sequence(x[i].cpu())
    reconstructed = decode_sequence(recon[i].cpu())
    
    # Calculate match percentage
    match = (x[i].cpu() == recon[i].cpu()).float().mean().item()
    
    print(f"Original:      {original}")
    print(f"Reconstructed: {reconstructed}")
    print(f"Match: {match*100:.1f}%\n")

## 7. Latent Space Interpolation

One powerful property of VAEs is the ability to interpolate between sequences in latent space.

In [ ]:
def interpolate_sequences(model, seq1_idx, seq2_idx, n_steps=5):
    """
    Interpolate between two sequences in latent space.
    """
    model.eval()
    
    with torch.no_grad():
        # Encode both sequences
        mu1, _ = model.encode(seq1_idx.unsqueeze(0).to(device))
        mu2, _ = model.encode(seq2_idx.unsqueeze(0).to(device))
        
        # Interpolate
        interpolations = []
        for alpha in np.linspace(0, 1, n_steps):
            z = (1 - alpha) * mu1 + alpha * mu2
            logits = model.decode(z)
            seq_idx = torch.argmax(logits, dim=-1)
            interpolations.append(seq_idx[0].cpu())
    
    return interpolations

# Get two sequences from different families
family_0_idx = [i for i, l in enumerate(val_labels) if l == 0][0]
family_2_idx = [i for i, l in enumerate(val_labels) if l == 2][0]

seq1 = val_dataset[family_0_idx][0]
seq2 = val_dataset[family_2_idx][0]

interpolations = interpolate_sequences(model, seq1, seq2, n_steps=7)

print("Interpolation from Family 0 to Family 2:\n")
for i, interp in enumerate(interpolations):
    alpha = i / (len(interpolations) - 1)
    seq = decode_sequence(interp)
    print(f"alpha={alpha:.2f}: {seq}")

## 8. Beta-VAE: Effect of KL Weight

Let's explore how changing beta affects the latent space structure.

In [ ]:
def train_vae_with_beta(beta, n_epochs=30):
    """Train a VAE with specific beta value"""
    model = ProteinVAE(
        seq_length=SEQ_LENGTH,
        vocab_size=VOCAB_SIZE,
        hidden_dim=256,
        latent_dim=16
    ).to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    
    for epoch in range(n_epochs):
        train_epoch(model, train_loader, optimizer, beta)
    
    # Get latent representations
    latents, labels = get_latent_representations(model, val_loader)
    
    # Final evaluation
    _, _, kl, acc = evaluate(model, val_loader, beta)
    
    return model, latents, labels, kl, acc

# Compare different beta values
betas = [0.1, 1.0, 4.0]
results = {}

print("Training VAEs with different beta values...\n")
for beta in betas:
    print(f"Training beta={beta}...")
    model, latents, labels, kl, acc = train_vae_with_beta(beta)
    results[beta] = {
        'latents': latents,
        'labels': labels,
        'kl': kl,
        'accuracy': acc
    }
    print(f"  KL: {kl:.4f}, Accuracy: {acc:.4f}\n")

In [ ]:
# Visualize latent spaces for different betas
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, beta in zip(axes, betas):
    latents = results[beta]['latents']
    labels = results[beta]['labels']
    
    # t-SNE
    if latents.shape[1] > 2:
        tsne = TSNE(n_components=2, random_state=42, perplexity=30)
        latents_2d = tsne.fit_transform(latents)
    else:
        latents_2d = latents
    
    scatter = ax.scatter(latents_2d[:, 0], latents_2d[:, 1], 
                        c=labels, cmap='viridis', alpha=0.7, s=20)
    ax.set_title(f'Beta = {beta}\nKL: {results[beta]["kl"]:.2f}, Acc: {results[beta]["accuracy"]:.2f}')
    ax.set_xlabel('t-SNE 1')
    ax.set_ylabel('t-SNE 2')

plt.suptitle('Effect of Beta on Latent Space Structure', fontsize=14)
plt.tight_layout()
plt.show()

## 9. Summary and Key Observations

### What we learned:

1. **VAE Architecture**: Encoder maps to distribution parameters, decoder reconstructs from samples

2. **Reparameterization Trick**: Enables gradient flow through sampling: $z = \mu + \sigma \cdot \epsilon$

3. **ELBO Loss**: Balances reconstruction quality and latent space regularity
   - Lower beta: Better reconstruction, less structured latent space
   - Higher beta: More structured latent space, worse reconstruction

4. **Latent Space Properties**:
   - Sequences from the same family cluster together
   - Smooth interpolation between sequences is possible
   - Random sampling from prior generates valid-looking sequences

### Limitations:
- VAEs tend to produce "blurry" outputs (average over modes)
- May not capture fine-grained sequence details
- Latent space quality depends heavily on training data

### Next steps:
- Try with real protein family data (e.g., from Pfam)
- Add conditional generation (CVAE) for property-guided design
- Compare with diffusion models for generation quality

In [ ]:
# Final summary statistics
print("=" * 50)
print("TRAINING SUMMARY")
print("=" * 50)
print(f"Model: ProteinVAE")
print(f"Sequence length: {SEQ_LENGTH}")
print(f"Vocabulary size: {VOCAB_SIZE}")
print(f"Latent dimension: {model.latent_dim}")
print(f"Final validation accuracy: {history['val_acc'][-1]:.4f}")
print(f"Final reconstruction loss: {history['val_recon'][-1]:.4f}")
print(f"Final KL divergence: {history['val_kl'][-1]:.4f}")